# Scraping from https://www.fotmob.com/it/leagues/55

In [123]:
import json

with open(r".\Datasets\Raw_JSONs\seriea_fixtures_2526.json", "r") as f:
    data = json.load(f)

# Inspect the top-level keys
print("Top keys:", list(data.keys()))
if "fixtures" in data:
    print("Fixtures key exists directly.")
elif "matches" in data:
    print("Matches key exists:", list(data["matches"].keys()) if isinstance(data["matches"], dict) else type(data["matches"]))
else:
    # Let's search recursively or print elements
    for k, v in data.items():
        if isinstance(v, dict):
            print(f"Key {k} subkeys:", list(v.keys())[:10])
        elif isinstance(v, list):
            print(f"Key {k} list length:", len(v))

Top keys: ['tabs', 'allAvailableSeasons', 'details', 'seostr', 'QAData', 'table', 'transfers', 'overview', 'stats', 'fixtures', 'playoff', 'seasons']
Fixtures key exists directly.


In [ ]:
# Let's inspect the 'fixtures' key
fixtures = data['fixtures']
print("Type of fixtures:", type(fixtures))
if isinstance(fixtures, dict):
    print("Fixtures dict keys:", list(fixtures.keys()))
elif isinstance(fixtures, list):
    print("Fixtures list length:", len(fixtures))
    if len(fixtures) > 0:
        print("First item sample:", json.dumps(fixtures[0], indent=2)[:500])

Type of fixtures: <class 'dict'>
Fixtures dict keys: ['firstUnplayedMatch', 'allMatches', 'hasOngoingMatch', 'fixtureInfo']


In [125]:
all_matches = data['fixtures']['allMatches']
print("Total matches in allMatches:", len(all_matches))
if len(all_matches) > 0:
    print("Match sample:", json.dumps(all_matches[0], indent=2))

Total matches in allMatches: 380
Match sample: {
  "round": "1",
  "roundName": 1,
  "pageUrl": "/matches/lecce-vs-genoa/3cj5b9#4803040",
  "id": "4803040",
  "home": {
    "name": "Genoa",
    "shortName": "Genoa",
    "id": "10233"
  },
  "away": {
    "name": "Lecce",
    "shortName": "Lecce",
    "id": "9888"
  },
  "status": {
    "utcTime": "2025-08-23T16:30:00Z",
    "timezone": "",
    "finished": true,
    "started": true,
    "cancelled": false,
    "awarded": false,
    "scoreStr": "0 - 0",
    "reason": {
      "short": "FT",
      "shortKey": "fulltime_short",
      "long": "Full-Time",
      "longKey": "finished"
    }
  }
}


# Now that we know where to extract data, we can extract what we need and save it in an Excel file

In [ ]:
import pandas as pd
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

matches_list = []
for m in all_matches:
    round_num = m.get("round", "")
    match_id = m.get("id", "")
    page_url = m.get("pageUrl", "")
    full_url = f"https://www.fotmob.com{page_url}" if page_url else ""
    
    home_name = m.get("home", {}).get("name", "")
    home_id = m.get("home", {}).get("id", "")
    away_name = m.get("away", {}).get("name", "")
    away_id = m.get("away", {}).get("id", "")
    
    status_obj = m.get("status", {})
    utc_time = status_obj.get("utcTime", "")
    score_str = status_obj.get("scoreStr", "")
    
    finished = status_obj.get("finished", False)
    cancelled = status_obj.get("cancelled", False)

    home_goals = ""
    away_goals = ""
    if score_str and " - " in score_str:
        try:
            parts = score_str.split(" - ")
            home_goals = int(parts[0])
            away_goals = int(parts[1])
        except:
            pass
            
    matches_list.append({
        "Giornata": int(round_num) if round_num.isdigit() else round_num,
        "ID Partita": match_id,
        "Data (UTC)": utc_time.replace("T", " ").replace("Z", ""),
        "Squadra Casa": home_name,
        "Squadra Fuori": away_name,
        "Risultato": score_str,
        "Gol Casa": home_goals,
        "Gol Fuori": away_goals,
        "Stato": "Finito" if finished else ("Cancellato" if cancelled else "Da giocare"),
        "URL FotMob": full_url
    })

## Check that data are saved correctly 

In [127]:
for i in range(35, 40):
    print(matches_list[i], "\n")

{'Giornata': 4, 'ID Partita': '4803072', 'Data (UTC)': '2025-09-21 13:00:00', 'Squadra Casa': 'Cremonese', 'Squadra Fuori': 'Parma', 'Risultato': '0 - 0', 'Gol Casa': 0, 'Gol Fuori': 0, 'Stato': 'Finito', 'URL FotMob': 'https://www.fotmob.com/matches/cremonese-vs-parma/2o48wh#4803072'} 

{'Giornata': 4, 'ID Partita': '4803068', 'Data (UTC)': '2025-09-21 13:00:00', 'Squadra Casa': 'Torino', 'Squadra Fuori': 'Atalanta', 'Risultato': '0 - 3', 'Gol Casa': 0, 'Gol Fuori': 3, 'Stato': 'Finito', 'URL FotMob': 'https://www.fotmob.com/matches/atalanta-vs-torino/2s0bp4#4803068'} 

{'Giornata': 4, 'ID Partita': '4803073', 'Data (UTC)': '2025-09-21 16:00:00', 'Squadra Casa': 'Fiorentina', 'Squadra Fuori': 'Como', 'Risultato': '1 - 2', 'Gol Casa': 1, 'Gol Fuori': 2, 'Stato': 'Finito', 'URL FotMob': 'https://www.fotmob.com/matches/fiorentina-vs-como/2w6bnm#4803073'} 

{'Giornata': 4, 'ID Partita': '4803074', 'Data (UTC)': '2025-09-21 18:45:00', 'Squadra Casa': 'Inter', 'Squadra Fuori': 'Sassuolo', '

## Now we create a new df to then save it as an Excel file 

In [128]:
df = pd.DataFrame(matches_list)
# Sort by round and then date/id
df = df.sort_values(by=["Giornata", "Data (UTC)"]).reset_index(drop=True)

In [129]:
print(df.head())

   Giornata ID Partita           Data (UTC) Squadra Casa Squadra Fuori  \
0         1    4803040  2025-08-23 16:30:00        Genoa         Lecce   
1         1    4803045  2025-08-23 16:30:00     Sassuolo        Napoli   
2         1    4803043  2025-08-23 18:45:00        Milan     Cremonese   
3         1    4803044  2025-08-23 18:45:00         Roma       Bologna   
4         1    4803038  2025-08-24 16:30:00     Cagliari    Fiorentina   

  Risultato  Gol Casa  Gol Fuori   Stato  \
0     0 - 0         0          0  Finito   
1     0 - 2         0          2  Finito   
2     1 - 2         1          2  Finito   
3     1 - 0         1          0  Finito   
4     1 - 1         1          1  Finito   

                                          URL FotMob  
0  https://www.fotmob.com/matches/lecce-vs-genoa/...  
1  https://www.fotmob.com/matches/sassuolo-vs-nap...  
2  https://www.fotmob.com/matches/cremonese-vs-mi...  
3  https://www.fotmob.com/matches/roma-vs-bologna...  
4  https://www.

In [130]:
path = r".\Datasets\Matches\serieA_fixtures_2526.xlsx"
df.to_excel(path, index=False)

## Style for Readability

In [131]:
#creazione workbook con openpyxl
wb = openpyxl.Workbook()

ws_data = wb.active
ws_data.title = "Calendario e Risultati"
ws_data.views.sheetView[0].showGridLines = True

In [132]:
# Stile pagina
navy_fill = PatternFill(start_color="1B365D", end_color="1B365D", fill_type="solid")
light_blue_fill = PatternFill(start_color="F2F5F9", end_color="F2F5F9", fill_type="solid")
white_font = Font(name="Segoe UI", size=11, bold=True, color="FFFFFF")
regular_font = Font(name="Segoe UI", size=10)
bold_font = Font(name="Segoe UI", size=10, bold=True)
title_font = Font(name="Segoe UI", size=16, bold=True, color="1B365D")
section_font = Font(name="Segoe UI", size=12, bold=True, color="1B365D")

thin_gray = Side(style='thin', color='D3D3D3')
cell_border = Border(left=thin_gray, right=thin_gray, top=thin_gray, bottom=thin_gray)

In [ ]:
# Populate Data Sheet
headers = list(df.columns)
ws_data.append(headers)
for row in ws_data[1]:
    row.fill = navy_fill
    row.font = white_font
    row.alignment = Alignment(horizontal="center", vertical="center")

for idx, r_data in df.iterrows():
    row_values = [r_data[h] for h in headers]
    ws_data.append(row_values)
    row_num = idx + 2
    
    # Apply link to URL FotMob column
    url_val = r_data["URL FotMob"]
    if url_val:
        cell = ws_data.cell(row=row_num, column=headers.index("URL FotMob") + 1)
        cell.hyperlink = url_val
        cell.font = Font(name="Segoe UI", size=10, color="0000FF", underline="single")
    
    # zebra striping
    if row_num % 2 == 1:
        for col_idx in range(1, len(headers) + 1):
            c = ws_data.cell(row=row_num, column=col_idx)
            if col_idx != headers.index("URL FotMob") + 1:
                c.fill = light_blue_fill

    # Borders and alignment
    for col_idx in range(1, len(headers) + 1):
        c = ws_data.cell(row=row_num, column=col_idx)
        c.border = cell_border
        if col_idx in [1, 2, 3, 6, 7, 8, 9]:
            c.alignment = Alignment(horizontal="center", vertical="center")
        else:
            c.alignment = Alignment(horizontal="left", vertical="center")

# Auto-adjust column widths
for col in ws_data.columns:
    max_len = max(len(str(cell.value or '')) for cell in col)
    col_letter = get_column_letter(col[0].column)
    ws_data.column_dimensions[col_letter].width = max(max_len + 3, 12)

In [134]:
ws_data.freeze_panes = "A2"
path_styled = r".\Datasets\Matches\Styled_SerieA_2526_Fixture_Stats.xlsx"
wb.save(path_styled)

# Now let's save the detailed statistics for each match
### Thanks to the match code obtained in the previous section, combined with the public API link, I download all the JSON files containing the detailed match statistics

In [135]:
# link API: https://www.fotmob.com/api/data/matchDetails?matchId={matchId}

In [136]:
print(headers)

['Giornata', 'ID Partita', 'Data (UTC)', 'Squadra Casa', 'Squadra Fuori', 'Risultato', 'Gol Casa', 'Gol Fuori', 'Stato', 'URL FotMob']


In [137]:
matchId = df["ID Partita"]
print(matchId)

0      4803040
1      4803045
2      4803043
3      4803044
4      4803038
        ...   
375    4803413
376    4803417
377    4803412
378    4803409
379    4803416
Name: ID Partita, Length: 380, dtype: str


In [ ]:
import os
import requests


OUTPUT_DIR = r".\Datasets\MatchStatsJSON_2526"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [139]:
def clean_header_string(text):
    if not text:
        return ""
    text = text.replace('\u2026', '...')
    return text.encode('latin-1', 'ignore').decode('latin-1')

## IMPORTANT
A new cookie needs to be inserted every time because it expires after a while; you can get it by capturing the network packets on the site using the Inspect tool

In [140]:
BASE_URL = "https://www.fotmob.com/api/data/matchDetails"
cookie_sporco = r"""u:location=%7B%22countryCode%22%3A%22IT%22%2C%22regionId%22%3A%2225%22%2C%22ip%22%3A%22127.0.0.1%22%2C%22ccode3%22%3A%22ITA%22%2C%22ccode3NoRegion%22%3A%22ITA%22%2C%22timezone%22%3A%22Europe%2FRome%22%7D; turnstile_verified=1.1779710658.4066c25e20662325be53b4b89e6498946c5d8aa81081e2ca802202e9771f4890; g_state={"i_l":1,"i_ll":1779719785159,"i_b":"LXcHBMBxPh75VOgx+zhLz6nes1pg6xL1sfb2Zpq8jQA","i_e":{"enable_itp_optimization":0},"i_et":1779710661468,"i_p":1779719854953}; _hjSessionUser_2585474=eyJpZCI6ImNlMmRkO…3MDIsImV4aXN0aW5nIjpmYWxzZX0=; NEXT_LOCALE=it; _hjSession_2585474=eyJpZCI6IjM2ODA2NWRlLWNmNWEtNDAxMS1iODYwLTJkN2M3M2FmOWE2NiIsImMiOjE3Nzk3MTk3NzQxNTIsInMiOjAsInIiOjAsInNiIjowLCJzciI6MCwic2UiOjAsImZzIjowLCJzcCI6MH0=; FCNEC=%5B%5B%22AKsRol_JazOlt3wnPP8hB56Z7xGcbtM_HBTkZ28nX8xBO9K9B9g4pT3J5wFZDoN00CUJu-YqLexWhCz9DMLKrpT49sqhibcFHW3Nfw7-eRI6e7pmwKl_VJk0uJsSV0S3tnaLiXto8iv0MDZ6mDysXE645IBUeaMq2g%3D%3D%22%5D%5D; _ga_G0V1WDW9B2=GS2.1.s1779719784$o1$g0$t1779719784$j60$l0$h728376776; _ga=GA1.1.1226782528.1779719785"""
cookie_pulito = clean_header_string(cookie_sporco)

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "it-IT,it;q=0.9,en-US;q=0.8,en;q=0.7",
    "Alt-Used": "www.fotmob.com",
    "Cookie": cookie_pulito
}

In [141]:
# Funzione per pulire i nomi delle squadre in quanto alcune potrebbero contnere caratteri non validi per i nomi dei file es: Hellas Verona -> Hellas_Verona
def clean_filename(text):
    text_clean = str(text).replace(" ", "_")
    return "".join(c for c in text_clean if c.isalnum() or c in ['_', '-']).strip()

In [ ]:
import time

print("\n ============== INIZIO DOWNLOAD FILE JSON ==============")
for index, row in df.iterrows():
    match_id = str(row['ID Partita'])
    giornata = str(row['Giornata'])
    home_team = clean_filename(row['Squadra Casa'])
    away_team = clean_filename(row['Squadra Fuori'])
    

    full_date = clean_filename(str(row['Data (UTC)']))
    
    
    # Formato per salvare i file secondo l'ordine del file Excel
    file_name = f"{giornata}_{full_date}_{match_id}_{home_team}_{away_team}.json"
    file_path = os.path.join(OUTPUT_DIR, file_name)

    print(f"[{index + 1}/{len(df)}] Scaricamento: {home_team} vs {away_team} (ID: {match_id})")
    
    # Richiesta HTTP
    params = {"matchId": match_id}
    try:
        response = requests.get(BASE_URL, params=params, headers=headers)
        
        if response.status_code == 200:
            match_json = response.json()
            
            # Salvataggio del file JSON
            with open(file_path, "w", encoding="utf-8") as out_file:
                import json
                json.dump(match_json, out_file, ensure_ascii=False, indent=4)
                
        elif response.status_code == 404:
            print(f"Dettagli non trovati (404) per il match {match_id}")
        else:
            print(f"Errore {response.status_code} per il match {match_id}")
            
    except Exception as e:
        print(f"Eccezione durante il download del match {match_id}: {e}")
        
    # Pausa di 1.5 secondi per evitare ban IP (per via delle tante richieste)
    time.sleep(1.5)

print("\n ============== FINE DOWNLOAD ==============")

## Now that I have downloaded all the files for the season, I can extract the statistics for each match
### First, let's start by studying the structure of the downloaded files

In [142]:
file_name = "1_2025-08-23_163000_4803040_Genoa_Lecce.json"
file_path = os.path.join(OUTPUT_DIR, file_name)

with open(file_path, "r", encoding="utf-8") as f:
    temp_data = json.load(f)

print("=== STRUTTURA DEI FILE MATCH ===")
print("Chiavi di primo livello:", list(temp_data.keys()))

# Ho gia visto la composizione del file, quindi controllo che ci siano le chiavi che mi servono (content -> lineup e stats)
if "content" in temp_data:
    print("\nSub-chiavi dentro 'content':", list(temp_data["content"].keys()))
    
    # Controllo per lineup e stats
    if "lineup" in temp_data["content"]:
        print("Presente 'lineup' che contiene dati e voti dei singoli giocatori")
    if "stats" in temp_data["content"]:
        print("Presente 'stats' che contiene statistiche generali di squadra")


=== STRUTTURA DEI FILE MATCH ===
Chiavi di primo livello: ['general', 'header', 'nav', 'ongoing', 'hasPendingVAR', 'content', 'seo']

Sub-chiavi dentro 'content': ['matchFacts', 'liveticker', 'superlive', 'buzz', 'stats', 'playerStats', 'shotmap', 'weather', 'lineup', 'hasPlayoff', 'table', 'h2h', 'momentum']
Presente 'lineup' che contiene dati e voti dei singoli giocatori
Presente 'stats' che contiene statistiche generali di squadra


In [143]:
print("=== STRUTTURA DI LINEUP ===")
print("Chiavi presenti in lineup:", list(temp_data["content"]["lineup"].keys()))

=== STRUTTURA DI LINEUP ===
Chiavi presenti in lineup: ['matchId', 'lineupType', 'source', 'availableFilters', 'homeTeam', 'awayTeam']


In [ ]:
print("=== STRUTTURA DI STATS ===")
print("Chiavi presenti in stats:", list(temp_data["content"]["stats"].keys()))
print("Sottochiavi presenti in stats -> Periods:", list(temp_data["content"]["stats"]["Periods"].keys()))
print("Sottochiavi presenti in stats -> Periods -> All:", list(temp_data["content"]["stats"]["Periods"]["All"].keys()))

In [144]:
stats = temp_data["content"]["stats"]["Periods"]["All"]["stats"]
print("Total unique stats in stats:", len(stats))
if len(stats) > 0:
    print("Stat sample:", json.dumps(stats[0], indent=2))

Total unique stats in stats: 7
Stat sample: {
  "title": "Top stats",
  "key": "top_stats",
  "stats": [
    {
      "title": "Ball possession",
      "key": "BallPossesion",
      "stats": [
        59,
        41
      ],
      "format": "integer",
      "type": "graph",
      "highlighted": "home"
    },
    {
      "title": "Expected goals (xG)",
      "key": "expected_goals",
      "stats": [
        "0.50",
        "0.37"
      ],
      "format": "double",
      "type": "text",
      "highlighted": "home"
    },
    {
      "title": "Total shots",
      "key": "total_shots",
      "stats": [
        5,
        7
      ],
      "format": "integer",
      "type": "text",
      "highlighted": "away"
    },
    {
      "title": "Shots on target",
      "key": "ShotsOnTarget",
      "stats": [
        2,
        0
      ],
      "format": "integer",
      "type": "text",
      "highlighted": "home"
    },
    {
      "title": "Touches in opposition box",
      "key": "touches_opp_box"

In [ ]:
dati_partita = {}

try:
        periods_stats = temp_data["content"]["stats"]["Periods"]["All"]["stats"]

        for category in periods_stats:
            sub_stats = category.get("stats", [])

            for stat in sub_stats:
                stat_title = stat.get("title")  # es. "Ball possession"
                values = stat.get("stats", [None, None])  # [Casa, Fuori]

                # Creiamo due colonne per differenziare i dati di Casa e Fuori
                dati_partita[f"{stat_title}_Casa"] = values[0]
                dati_partita[f"{stat_title}_Ospiti"] = values[1]

except KeyError:
    pass

print(dati_partita)

{'Ball possession_Casa': 59, 'Ball possession_Ospiti': 41, 'Expected goals (xG)_Casa': '0.50', 'Expected goals (xG)_Ospiti': '0.37', 'Total shots_Casa': 5, 'Total shots_Ospiti': 7, 'Shots on target_Casa': 2, 'Shots on target_Ospiti': 0, 'Touches in opposition box_Casa': 6, 'Touches in opposition box_Ospiti': 16, 'Big chances_Casa': 0, 'Big chances_Ospiti': 1, 'Big chances missed_Casa': 0, 'Big chances missed_Ospiti': 1, 'Accurate passes_Casa': '327 (75%)', 'Accurate passes_Ospiti': '207 (70%)', 'Yellow cards_Casa': 2, 'Yellow cards_Ospiti': 3, 'Corners_Casa': 3, 'Corners_Ospiti': 7, 'Shots_Casa': None, 'Shots_Ospiti': None, 'Shots off target_Casa': 2, 'Shots off target_Ospiti': 3, 'Blocked shots_Casa': 1, 'Blocked shots_Ospiti': 4, 'Hit woodwork_Casa': 0, 'Hit woodwork_Ospiti': 0, 'Shots inside box_Casa': 2, 'Shots inside box_Ospiti': 5, 'Shots outside box_Casa': 3, 'Shots outside box_Ospiti': 2, 'xG open play_Casa': '0.45', 'xG open play_Ospiti': '0.28', 'xG set play_Casa': '0.05', 'x

In [146]:
lista_sample = []
df_statistiche_sample = pd.DataFrame([dati_partita])
for i in range(len(df_statistiche_sample.columns)):
    print(df_statistiche_sample.columns[i], "--- --- --- --- ---" ,df_statistiche_sample.iloc[0, i])

Ball possession_Casa --- --- --- --- --- 59
Ball possession_Ospiti --- --- --- --- --- 41
Expected goals (xG)_Casa --- --- --- --- --- 0.50
Expected goals (xG)_Ospiti --- --- --- --- --- 0.37
Total shots_Casa --- --- --- --- --- 5
Total shots_Ospiti --- --- --- --- --- 7
Shots on target_Casa --- --- --- --- --- 2
Shots on target_Ospiti --- --- --- --- --- 0
Touches in opposition box_Casa --- --- --- --- --- 6
Touches in opposition box_Ospiti --- --- --- --- --- 16
Big chances_Casa --- --- --- --- --- 0
Big chances_Ospiti --- --- --- --- --- 1
Big chances missed_Casa --- --- --- --- --- 0
Big chances missed_Ospiti --- --- --- --- --- 1
Accurate passes_Casa --- --- --- --- --- 327 (75%)
Accurate passes_Ospiti --- --- --- --- --- 207 (70%)
Yellow cards_Casa --- --- --- --- --- 2
Yellow cards_Ospiti --- --- --- --- --- 3
Corners_Casa --- --- --- --- --- 3
Corners_Ospiti --- --- --- --- --- 7
Shots_Casa --- --- --- --- --- None
Shots_Ospiti --- --- --- --- --- None
Shots off target_Casa ---

# Now we can extract all the statistics from the JSON files and save them in a single dataset

In [ ]:
# lista vuota PRIMA del ciclo per accumulare i dati di tutte le partite
lista_partite_stats = []

print("============= INIZIO ELABORAZIONE TUTTI I FILE JSON ==============")

for index, row in df.iterrows():

    giornata = str(row["Giornata"])
    match_id = str(row["ID Partita"])
    full_date = str(row["Data (UTC)"])
    home_team = str(row["Squadra Casa"])
    away_team = str(row["Squadra Fuori"])
    result = str(row["Risultato"])
    home_goals = str(row['Gol Casa'])
    away_goals = str(row['Gol Fuori'])
    status = str(row["Stato"])
    link = str(row["URL FotMob"])

    clean_home_team = clean_filename(home_team)
    clean_away_team = clean_filename(away_team)
    clean_full_date = clean_filename(full_date)

    # Formato del nome dei file
    file_name = f"{giornata}_{clean_full_date}_{match_id}_{clean_home_team}_{clean_away_team}.json"
    file_path = os.path.join(OUTPUT_DIR, file_name)

    print("Elaborazione file:", file_name)

    # check che il file esista
    if not os.path.exists(file_path):
        print(f"File non trovato per il match {match_id}. Salto.")
        continue

    # Apertura del file
    with open(file_path, "r", encoding="utf-8") as f:
        temp_data = json.load(f)

    # Inizializziamo il dizionario della partita corrente con i dati gia presenti
    dati_partita = {
        "Giornata": giornata,
        "ID Partita": match_id,
        "Data (UTC)": full_date,
        "Squadra Casa": home_team,
        "Squadra Fuori": away_team,
        "Risultato": result,
        "Gol Casa": home_goals,
        "Gol Fuori": away_goals,
        "Stato": status,
        "URL FotMob": link,
    }

    # Estrazione delle statistiche di squadra
    try:
        periods_stats = temp_data["content"]["stats"]["Periods"]["All"]["stats"]

        for category in periods_stats:
            sub_stats = category.get("stats", [])

            for stat in sub_stats:
                stat_title = stat.get("title")  # es. "Ball possession"
                values = stat.get("stats", [None, None])  # [Casa, Fuori]

                # Creiamo due colonne per differenziare i dati di Casa e Fuori
                dati_partita[f"{stat_title}_Casa"] = values[0]
                dati_partita[f"{stat_title}_Fuori"] = values[1]

    except KeyError:
        pass

    # Aggiungiamo le stats al totale
    lista_partite_stats.append(dati_partita)


print("============= FINE ELABORAZIONE TUTTI I FILE JSON ==============")

============= INIZIO ELABORAZIONE TUTTI I FILE JSON ==============
Elaborazione file: 1_2025-08-23_163000_4803040_Genoa_Lecce.json
Elaborazione file: 1_2025-08-23_163000_4803045_Sassuolo_Napoli.json
Elaborazione file: 1_2025-08-23_184500_4803043_Milan_Cremonese.json
Elaborazione file: 1_2025-08-23_184500_4803044_Roma_Bologna.json
Elaborazione file: 1_2025-08-24_163000_4803038_Cagliari_Fiorentina.json
Elaborazione file: 1_2025-08-24_163000_4803039_Como_Lazio.json
Elaborazione file: 1_2025-08-24_184500_4803037_Atalanta_Pisa.json
Elaborazione file: 1_2025-08-24_184500_4803042_Juventus_Parma.json
Elaborazione file: 1_2025-08-25_163000_4803046_Udinese_Hellas_Verona.json
Elaborazione file: 1_2025-08-25_184500_4803041_Inter_Torino.json
Elaborazione file: 2_2025-08-29_163000_4803048_Cremonese_Sassuolo.json
Elaborazione file: 2_2025-08-29_184500_4803052_Lecce_Milan.json
Elaborazione file: 2_2025-08-30_163000_4803047_Bologna_Como.json
Elaborazione file: 2_2025-08-30_163000_4803054_Parma_Atalanta

In [ ]:
# creo dataframe e lo salvo in excel
df_statistiche_totale = pd.DataFrame(lista_partite_stats)

path_stats = r".\Datasets\MatchStatsTotal\SerieA_2526_Fixture_Stats.xlsx"
df_statistiche_totale.to_excel(path_stats, index=False)  

## Style for Readability

In [150]:
#creazione workbook con openpyxl
wb = openpyxl.Workbook()

ws_data = wb.active
ws_data.title = "Risultati e Statistiche"
ws_data.views.sheetView[0].showGridLines = True

In [151]:
# Stile pagina
navy_fill = PatternFill(start_color="1B365D", end_color="1B365D", fill_type="solid")
light_blue_fill = PatternFill(start_color="F2F5F9", end_color="F2F5F9", fill_type="solid")
white_font = Font(name="Segoe UI", size=11, bold=True, color="FFFFFF")
regular_font = Font(name="Segoe UI", size=10)
bold_font = Font(name="Segoe UI", size=10, bold=True)
title_font = Font(name="Segoe UI", size=16, bold=True, color="1B365D")
section_font = Font(name="Segoe UI", size=12, bold=True, color="1B365D")

thin_gray = Side(style='thin', color='D3D3D3')
cell_border = Border(left=thin_gray, right=thin_gray, top=thin_gray, bottom=thin_gray)

In [ ]:
# Populate Data Sheet
headers = list(df_statistiche_totale.columns)
ws_data.append(headers)
for row in ws_data[1]:
    row.fill = navy_fill
    row.font = white_font
    row.alignment = Alignment(horizontal="center", vertical="center")

for idx, r_data in df_statistiche_totale.iterrows():
    row_values = [r_data[h] for h in headers]
    ws_data.append(row_values)
    row_num = idx + 2
    
    # Apply link to URL FotMob column
    url_val = r_data["URL FotMob"]
    if url_val:
        cell = ws_data.cell(row=row_num, column=headers.index("URL FotMob") + 1)
        cell.hyperlink = url_val
        cell.font = Font(name="Segoe UI", size=10, color="0000FF", underline="single")
    
    # Zebra striping
    if row_num % 2 == 1:
        for col_idx in range(1, len(headers) + 1):
            c = ws_data.cell(row=row_num, column=col_idx)
            if col_idx != headers.index("URL FotMob") + 1:
                c.fill = light_blue_fill

    # Borders and alignment
    for col_idx in range(1, len(headers) + 1):
        c = ws_data.cell(row=row_num, column=col_idx)
        c.border = cell_border
        if col_idx in [1, 2, 3, 6, 7, 8, 9]:
            c.alignment = Alignment(horizontal="center", vertical="center")
        else:
            c.alignment = Alignment(horizontal="left", vertical="center")

# Auto-adjust column widths
for col in ws_data.columns:
    max_len = max(len(str(cell.value or '')) for cell in col)
    col_letter = get_column_letter(col[0].column)
    ws_data.column_dimensions[col_letter].width = max(max_len + 3, 12)

In [ ]:
ws_data.freeze_panes = "A2"
path_stats_styled = r".\Datasets\MatchStatsTotal\Styled_SerieA_2526_Fixture_Stats.xlsx"
wb.save(path_stats_styled)

# The process for the other two seasons (2023/2024 e 2024/2025) is identical
If you want to use the requests library, you need to use these API links:
- https://www.fotmob.com/api/data/leagues?id=55&ccode3=ITA&season=2023%2F2024
- https://www.fotmob.com/api/data/leagues?id=55&ccode3=ITA&season=2024%2F2025

For the most recent season, you need to use this API link:
- https://www.fotmob.com/api/data/leagues?id=55&ccode3=ITA

## 2024/2025

In [155]:
with open(r".\Datasets\Raw_JSONs\seriea_fixtures_2425.json", "r") as f:
    data = json.load(f)

# Inspect the top-level keys
print("Top keys:", list(data.keys()))
if "fixtures" in data:
    print("Fixtures key exists directly.")
elif "matches" in data:
    print("Matches key exists:", list(data["matches"].keys()) if isinstance(data["matches"], dict) else type(data["matches"]))
else:
    # Let's search recursively or print elements
    for k, v in data.items():
        if isinstance(v, dict):
            print(f"Key {k} subkeys:", list(v.keys())[:10])
        elif isinstance(v, list):
            print(f"Key {k} list length:", len(v))
print("\n")


# Let's inspect the 'fixtures' key structure
fixtures = data['fixtures']
print("Type of fixtures:", type(fixtures))
if isinstance(fixtures, dict):
    print("Fixtures dict keys:", list(fixtures.keys()))
elif isinstance(fixtures, list):
    print("Fixtures list length:", len(fixtures))
    if len(fixtures) > 0:
        print("First item sample:", json.dumps(fixtures[0], indent=2)[:500])
print("\n")


all_matches = data['fixtures']['allMatches']
print("Total matches in allMatches:", len(all_matches))
if len(all_matches) > 0:
    print("Match sample:", json.dumps(all_matches[0], indent=2))

Top keys: ['tabs', 'allAvailableSeasons', 'details', 'seostr', 'QAData', 'table', 'transfers', 'overview', 'stats', 'fixtures', 'playoff', 'seasons']
Fixtures key exists directly.


Type of fixtures: <class 'dict'>
Fixtures dict keys: ['firstUnplayedMatch', 'allMatches', 'hasOngoingMatch', 'fixtureInfo']


Total matches in allMatches: 380
Match sample: {
  "round": "1",
  "roundName": 1,
  "pageUrl": "/matches/inter-vs-genoa/2xzyqb#4535247",
  "id": "4535247",
  "home": {
    "name": "Genoa",
    "shortName": "Genoa",
    "id": "10233"
  },
  "away": {
    "name": "Inter",
    "shortName": "Inter",
    "id": "8636"
  },
  "status": {
    "utcTime": "2024-08-17T16:30:00Z",
    "timezone": "",
    "finished": true,
    "started": true,
    "cancelled": false,
    "awarded": false,
    "scoreStr": "2 - 2",
    "reason": {
      "short": "FT",
      "shortKey": "fulltime_short",
      "long": "Full-Time",
      "longKey": "finished"
    }
  }
}


In [ ]:
matches_list = []
for m in all_matches:
    round_num = m.get("round", "")
    match_id = m.get("id", "")
    page_url = m.get("pageUrl", "")
    full_url = f"https://www.fotmob.com{page_url}" if page_url else ""
    
    home_name = m.get("home", {}).get("name", "")
    home_id = m.get("home", {}).get("id", "")
    away_name = m.get("away", {}).get("name", "")
    away_id = m.get("away", {}).get("id", "")
    
    status_obj = m.get("status", {})
    utc_time = status_obj.get("utcTime", "")
    score_str = status_obj.get("scoreStr", "")
    
    finished = status_obj.get("finished", False)
    cancelled = status_obj.get("cancelled", False)

    home_goals = ""
    away_goals = ""
    if score_str and " - " in score_str:
        try:
            parts = score_str.split(" - ")
            home_goals = int(parts[0])
            away_goals = int(parts[1])
        except:
            pass
            
    matches_list.append({
        "Giornata": int(round_num) if round_num.isdigit() else round_num,
        "ID Partita": match_id,
        "Data (UTC)": utc_time.replace("T", " ").replace("Z", ""),
        "Squadra Casa": home_name,
        "Squadra Fuori": away_name,
        "Risultato": score_str,
        "Gol Casa": home_goals,
        "Gol Fuori": away_goals,
        "Stato": "Finito" if finished else ("Cancellato" if cancelled else "Da giocare"),
        "URL FotMob": full_url
    })
print("\n")


for i in range(35, 40):
    print(matches_list[i], "\n")
print("\n")


df = pd.DataFrame(matches_list)

df = df.sort_values(by=["Giornata", "Data (UTC)"]).reset_index(drop=True)
print(df.head())
print("\n")


path = r".\Datasets\Matches\serieA_fixtures_2425.xlsx"
df.to_excel(path, index=False)



#creazione workbook con openpyxl
wb = openpyxl.Workbook()

ws_data = wb.active
ws_data.title = "Calendario e Risultati"
ws_data.views.sheetView[0].showGridLines = True



# Stile pagina
navy_fill = PatternFill(start_color="1B365D", end_color="1B365D", fill_type="solid")
light_blue_fill = PatternFill(start_color="F2F5F9", end_color="F2F5F9", fill_type="solid")
white_font = Font(name="Segoe UI", size=11, bold=True, color="FFFFFF")
regular_font = Font(name="Segoe UI", size=10)
bold_font = Font(name="Segoe UI", size=10, bold=True)
title_font = Font(name="Segoe UI", size=16, bold=True, color="1B365D")
section_font = Font(name="Segoe UI", size=12, bold=True, color="1B365D")

thin_gray = Side(style='thin', color='D3D3D3')
cell_border = Border(left=thin_gray, right=thin_gray, top=thin_gray, bottom=thin_gray)



# Populate Data Sheet
headers = list(df.columns)
ws_data.append(headers)
for row in ws_data[1]:
    row.fill = navy_fill
    row.font = white_font
    row.alignment = Alignment(horizontal="center", vertical="center")

for idx, r_data in df.iterrows():
    row_values = [r_data[h] for h in headers]
    ws_data.append(row_values)
    row_num = idx + 2
    
    # Apply link to URL FotMob column
    url_val = r_data["URL FotMob"]
    if url_val:
        cell = ws_data.cell(row=row_num, column=headers.index("URL FotMob") + 1)
        cell.hyperlink = url_val
        cell.font = Font(name="Segoe UI", size=10, color="0000FF", underline="single")
    
    # Zebra striping
    if row_num % 2 == 1:
        for col_idx in range(1, len(headers) + 1):
            c = ws_data.cell(row=row_num, column=col_idx)
            if col_idx != headers.index("URL FotMob") + 1:
                c.fill = light_blue_fill

    # Borders and alignment
    for col_idx in range(1, len(headers) + 1):
        c = ws_data.cell(row=row_num, column=col_idx)
        c.border = cell_border
        if col_idx in [1, 2, 3, 6, 7, 8, 9]:
            c.alignment = Alignment(horizontal="center", vertical="center")
        else:
            c.alignment = Alignment(horizontal="left", vertical="center")

# Auto-adjust column widths
for col in ws_data.columns:
    max_len = max(len(str(cell.value or '')) for cell in col)
    col_letter = get_column_letter(col[0].column)
    ws_data.column_dimensions[col_letter].width = max(max_len + 3, 12)



ws_data.freeze_panes = "A2"
path_styled = r".\Datasets\Matches\Styled_SerieA_2425_Fixture_Stats.xlsx"
wb.save(path_styled)



{'Giornata': 4, 'ID Partita': '4535289', 'Data (UTC)': '2024-09-15 13:00:00', 'Squadra Casa': 'Torino', 'Squadra Fuori': 'Lecce', 'Risultato': '0 - 0', 'Gol Casa': 0, 'Gol Fuori': 0, 'Stato': 'Finito', 'URL FotMob': 'https://www.fotmob.com/matches/torino-vs-lecce/37g3q6#4535289'} 

{'Giornata': 4, 'ID Partita': '4535281', 'Data (UTC)': '2024-09-15 16:00:00', 'Squadra Casa': 'Cagliari', 'Squadra Fuori': 'Napoli', 'Risultato': '0 - 4', 'Gol Casa': 0, 'Gol Fuori': 4, 'Stato': 'Finito', 'URL FotMob': 'https://www.fotmob.com/matches/cagliari-vs-napoli/2su8st#4535281'} 

{'Giornata': 4, 'ID Partita': '4535287', 'Data (UTC)': '2024-09-15 18:45:00', 'Squadra Casa': 'Monza', 'Squadra Fuori': 'Inter', 'Risultato': '1 - 1', 'Gol Casa': 1, 'Gol Fuori': 1, 'Stato': 'Finito', 'URL FotMob': 'https://www.fotmob.com/matches/monza-vs-inter/1w8tzq#4535287'} 

{'Giornata': 4, 'ID Partita': '4535288', 'Data (UTC)': '2024-09-16 16:30:00', 'Squadra Casa': 'Parma', 'Squadra Fuori': 'Udinese', 'Risultato': '

In [ ]:
# link API: https://www.fotmob.com/api/data/matchDetails?matchId={matchId}
print(headers)
print("\n")


matchId = df["ID Partita"]
print(matchId)
print("\n")


OUTPUT_DIR = r".\Datasets\MatchStatsJSON_2425"
os.makedirs(OUTPUT_DIR, exist_ok=True)



BASE_URL = "https://www.fotmob.com/api/data/matchDetails"
cookie_sporco = r"""g_state={"i_l":2,"i_ll":1779810410293,"i_b":"nRHE3uwL3TlzrPYvxMFcI0bZ+dl1r6S207EfAquto0Q","i_e":{"enable_itp_optimization":0},"i_et":1779710661468,"i_p":1779894495379}; _hjSessionUser_2585474=eyJpZCI6ImNlMmRkOTllLWE5NDMtNWFjYS05YzE3LTI2MTQxNTdhMTRlOSIsImNyZWF0ZWQiOjE3Nzk3MTA2NjE2MzQsImV4aXN0aW5nIjp0cnVlfQ==; FCCDCF=%5Bnull%2Cnull%2Cnull%2C%5B%22CQkw6oAQkw6oAEsACBITCgFoAP_gAEPgABpwLItB_C7dbWFDyLZ3absEeIhHx_hjasQwAAbAAmAFTBqQoJQGwmEYNAyCtCACGAAAKmTBIQIkCAAAAUAAIIAFIADMAAyQIBAIICBAiAEJAEBACEACCoAgUQCIgIJEFEQAm…rope%2FRome%22%7D; turnstile_verified=1.1779808090.d3c4b73a6c12d89300538a404f8022bc7f06d56f6306d6290bbe44a8a9fa453d; panoramaId_expiry=1779894492599; _hjSession_2585474=eyJpZCI6IjJkZjFlZjA1LTQ2NGItNDI4ZC1hYjAxLTdjOTMwODQzMDIxYyIsImMiOjE3Nzk4MTAzODQyODYsInMiOjAsInIiOjAsInNiIjowLCJzciI6MCwic2UiOjAsImZzIjowLCJzcCI6MX0=; FCNEC=%5B%5B%22AKsRol_4MENqJVHfam_1cJZuUm4Zg8gfe-eg-5W8vNrNNULYF96uETbLOxC-5Hg64DGPfoapHM-YN-5TSRXYQBMGcAJ_S-t8QowY8XoJ3jUYDjlXctL1CY5vfqJn3RcubAK4SnANnAe8ksOq_cBTCV9f51AQxRaYnA%3D%3D%22%5D%5D"""
cookie_pulito = clean_header_string(cookie_sporco)

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "it-IT,it;q=0.9,en-US;q=0.8,en;q=0.7",
    "Alt-Used": "www.fotmob.com",
    "Cookie": cookie_pulito
}



print("\n ============== INIZIO DOWNLOAD FILE JSON ==============")
for index, row in df.iterrows():
    match_id = str(row['ID Partita'])
    giornata = str(row['Giornata'])

    home_team = clean_filename(row['Squadra Casa'])
    away_team = clean_filename(row['Squadra Fuori'])
    
    
    full_date = clean_filename(str(row['Data (UTC)']))
    
    
    # Formato per salvare i file secondo l'ordine del file Excel
    file_name = f"{giornata}_{full_date}_{match_id}_{home_team}_{away_team}.json"
    file_path = os.path.join(OUTPUT_DIR, file_name)

    print(f"[{index + 1}/{len(df)}] Scaricamento: {home_team} vs {away_team} (ID: {match_id})")
    
    # Richiesta HTTP
    params = {"matchId": match_id}
    try:
        response = requests.get(BASE_URL, params=params, headers=headers)
        
        if response.status_code == 200:
            match_json = response.json()
            
            # Salvataggio del file JSON
            with open(file_path, "w", encoding="utf-8") as out_file:
                import json
                json.dump(match_json, out_file, ensure_ascii=False, indent=4)
                
        elif response.status_code == 404:
            print(f"Dettagli non trovati (404) per il match {match_id}")
        else:
            print(f"Errore {response.status_code} per il match {match_id}")
            
    except Exception as e:
        print(f"Eccezione durante il download del match {match_id}: {e}")
        
    # Pausa di 1.5 secondi per evitare ban IP (per via delle tante richieste)
    time.sleep(1.5)

print("\n ============== FINE DOWNLOAD ==============")

{'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36', 'Accept': 'application/json, text/plain, */*', 'Accept-Language': 'it-IT,it;q=0.9,en-US;q=0.8,en;q=0.7', 'Alt-Used': 'www.fotmob.com', 'Cookie': 'u:location=%7B%22countryCode%22%3A%22IT%22%2C%22regionId%22%3A%2225%22%2C%22ip%22%3A%22127.0.0.1%22%2C%22ccode3%22%3A%22ITA%22%2C%22ccode3NoRegion%22%3A%22ITA%22%2C%22timezone%22%3A%22Europe%2FRome%22%7D; turnstile_verified=1.1779710658.4066c25e20662325be53b4b89e6498946c5d8aa81081e2ca802202e9771f4890; g_state={"i_l":1,"i_ll":1779719785159,"i_b":"LXcHBMBxPh75VOgx+zhLz6nes1pg6xL1sfb2Zpq8jQA","i_e":{"enable_itp_optimization":0},"i_et":1779710661468,"i_p":1779719854953}; _hjSessionUser_2585474=eyJpZCI6ImNlMmRkO...3MDIsImV4aXN0aW5nIjpmYWxzZX0=; NEXT_LOCALE=it; _hjSession_2585474=eyJpZCI6IjM2ODA2NWRlLWNmNWEtNDAxMS1iODYwLTJkN2M3M2FmOWE2NiIsImMiOjE3Nzk3MTk3NzQxNTIsInMiOjAsInIiOjAsInNiIjowLCJzciI6MCwic2UiOjAsImZzIjowLCJzcCI

In [ ]:
# lista vuota PRIMA del ciclo per accumulare i dati di tutte le partite
lista_partite_stats = []

print("============= INIZIO ELABORAZIONE TUTTI I FILE JSON ==============")

for index, row in df.iterrows():

    giornata = str(row["Giornata"])
    match_id = str(row["ID Partita"])
    full_date = str(row["Data (UTC)"])
    home_team = str(row["Squadra Casa"])
    away_team = str(row["Squadra Fuori"])
    result = str(row["Risultato"])
    home_goals = str(row['Gol Casa'])
    away_goals = str(row['Gol Fuori'])
    status = str(row["Stato"])
    link = str(row["URL FotMob"])

    clean_home_team = clean_filename(home_team)
    clean_away_team = clean_filename(away_team)
    clean_full_date = clean_filename(full_date)

    # Formato del nome dei file
    file_name = f"{giornata}_{clean_full_date}_{match_id}_{clean_home_team}_{clean_away_team}.json"
    file_path = os.path.join(OUTPUT_DIR, file_name)

    print("Elaborazione file:", file_name)

    # check che il file esista
    if not os.path.exists(file_path):
        print(f"File non trovato per il match {match_id}. Salto.")
        continue

    # Apertura del file
    with open(file_path, "r", encoding="utf-8") as f:
        temp_data = json.load(f)

    # Inizializziamo il dizionario della partita corrente con i dati gia presenti
    dati_partita = {
        "Giornata": giornata,
        "ID Partita": match_id,
        "Data (UTC)": full_date,
        "Squadra Casa": home_team,
        "Squadra Fuori": away_team,
        "Risultato": result,
        "Gol Casa": home_goals,
        "Gol Fuori": away_goals,
        "Stato": status,
        "URL FotMob": link,
    }

    # Estrazione delle statistiche di squadra
    try:
        periods_stats = temp_data["content"]["stats"]["Periods"]["All"]["stats"]

        for category in periods_stats:
            sub_stats = category.get("stats", [])

            for stat in sub_stats:
                stat_title = stat.get("title")  # es. "Ball possession"
                values = stat.get("stats", [None, None])  # [Casa, Fuori]

                # Creiamo due colonne per differenziare i dati di Casa e Fuori
                dati_partita[f"{stat_title}_Casa"] = values[0]
                dati_partita[f"{stat_title}_Fuori"] = values[1]

    except KeyError:
        pass

    # Aggiungiamo le stats al totale
    lista_partite_stats.append(dati_partita)


print("============= FINE ELABORAZIONE TUTTI I FILE JSON ==============")




# creo dataframe e lo salvo in excel
df_statistiche_totale = pd.DataFrame(lista_partite_stats)

path_stats = r".\Datasets\MatchStatsTotal\SerieA_2425_Fixture_Stats.xlsx"
df_statistiche_totale.to_excel(path_stats, index=False)  




#creazione workbook con openpyxl
wb = openpyxl.Workbook()

ws_data = wb.active
ws_data.title = "Risultati e Statistiche"
ws_data.views.sheetView[0].showGridLines = True


# Stile pagina
navy_fill = PatternFill(start_color="1B365D", end_color="1B365D", fill_type="solid")
light_blue_fill = PatternFill(start_color="F2F5F9", end_color="F2F5F9", fill_type="solid")
white_font = Font(name="Segoe UI", size=11, bold=True, color="FFFFFF")
regular_font = Font(name="Segoe UI", size=10)
bold_font = Font(name="Segoe UI", size=10, bold=True)
title_font = Font(name="Segoe UI", size=16, bold=True, color="1B365D")
section_font = Font(name="Segoe UI", size=12, bold=True, color="1B365D")

thin_gray = Side(style='thin', color='D3D3D3')
cell_border = Border(left=thin_gray, right=thin_gray, top=thin_gray, bottom=thin_gray)


# Populate Data Sheet
headers = list(df_statistiche_totale.columns)
ws_data.append(headers)
for row in ws_data[1]:
    row.fill = navy_fill
    row.font = white_font
    row.alignment = Alignment(horizontal="center", vertical="center")

for idx, r_data in df_statistiche_totale.iterrows():
    row_values = [r_data[h] for h in headers]
    ws_data.append(row_values)
    row_num = idx + 2
    
    # Apply link to URL FotMob column
    url_val = r_data["URL FotMob"]
    if url_val:
        cell = ws_data.cell(row=row_num, column=headers.index("URL FotMob") + 1)
        cell.hyperlink = url_val
        cell.font = Font(name="Segoe UI", size=10, color="0000FF", underline="single")
    
    # Zebra striping
    if row_num % 2 == 1:
        for col_idx in range(1, len(headers) + 1):
            c = ws_data.cell(row=row_num, column=col_idx)
            if col_idx != headers.index("URL FotMob") + 1:
                c.fill = light_blue_fill

    # Borders and alignment
    for col_idx in range(1, len(headers) + 1):
        c = ws_data.cell(row=row_num, column=col_idx)
        c.border = cell_border
        if col_idx in [1, 2, 3, 6, 7, 8, 9]:
            c.alignment = Alignment(horizontal="center", vertical="center")
        else:
            c.alignment = Alignment(horizontal="left", vertical="center")

# Auto-adjust column widths
for col in ws_data.columns:
    max_len = max(len(str(cell.value or '')) for cell in col)
    col_letter = get_column_letter(col[0].column)
    ws_data.column_dimensions[col_letter].width = max(max_len + 3, 12)


ws_data.freeze_panes = "A2"
path_stats_styled = r".\Datasets\MatchStatsTotal\Styled_SerieA_2425_Fixture_Stats.xlsx"
wb.save(path_stats_styled)

============= INIZIO ELABORAZIONE TUTTI I FILE JSON ==============
Elaborazione file: 1_2024-08-17_163000_4535247_Genoa_Inter.json
Elaborazione file: 1_2024-08-17_163000_4535253_Parma_Fiorentina.json
Elaborazione file: 1_2024-08-17_184500_4535252_Milan_Torino.json
Elaborazione file: 1_2024-08-17_184500_4535246_Empoli_Monza.json
Elaborazione file: 1_2024-08-18_163000_4535244_Bologna_Udinese.json
Elaborazione file: 1_2024-08-18_163000_4535248_Hellas_Verona_Napoli.json
Elaborazione file: 1_2024-08-18_184500_4535245_Cagliari_Roma.json
Elaborazione file: 1_2024-08-18_184500_4535250_Lazio_Venezia.json
Elaborazione file: 1_2024-08-19_163000_4535251_Lecce_Atalanta.json
Elaborazione file: 1_2024-08-19_184500_4535249_Juventus_Como.json
Elaborazione file: 2_2024-08-24_163000_4535266_Parma_Milan.json
Elaborazione file: 2_2024-08-24_163000_4535269_Udinese_Lazio.json
Elaborazione file: 2_2024-08-24_184500_4535263_Inter_Lecce.json
Elaborazione file: 2_2024-08-24_184500_4535264_Monza_Genoa.json
Elabor

## 2023/2024

In [161]:
with open(r".\Datasets\Raw_JSONs\seriea_fixtures_2324.json", "r") as f:
    data = json.load(f)

# Inspect the top-level keys
print("Top keys:", list(data.keys()))
if "fixtures" in data:
    print("Fixtures key exists directly.")
elif "matches" in data:
    print("Matches key exists:", list(data["matches"].keys()) if isinstance(data["matches"], dict) else type(data["matches"]))
else:
    # Let's search recursively or print elements
    for k, v in data.items():
        if isinstance(v, dict):
            print(f"Key {k} subkeys:", list(v.keys())[:10])
        elif isinstance(v, list):
            print(f"Key {k} list length:", len(v))
print("\n")


# Let's inspect the 'fixtures' key structure
fixtures = data['fixtures']
print("Type of fixtures:", type(fixtures))
if isinstance(fixtures, dict):
    print("Fixtures dict keys:", list(fixtures.keys()))
elif isinstance(fixtures, list):
    print("Fixtures list length:", len(fixtures))
    if len(fixtures) > 0:
        print("First item sample:", json.dumps(fixtures[0], indent=2)[:500])
print("\n")


all_matches = data['fixtures']['allMatches']
print("Total matches in allMatches:", len(all_matches))
if len(all_matches) > 0:
    print("Match sample:", json.dumps(all_matches[0], indent=2))

Top keys: ['tabs', 'allAvailableSeasons', 'details', 'seostr', 'QAData', 'table', 'transfers', 'overview', 'stats', 'fixtures', 'playoff', 'seasons']
Fixtures key exists directly.


Type of fixtures: <class 'dict'>
Fixtures dict keys: ['firstUnplayedMatch', 'allMatches', 'hasOngoingMatch', 'fixtureInfo']


Total matches in allMatches: 380
Match sample: {
  "round": "1",
  "roundName": 1,
  "pageUrl": "/matches/empoli-vs-hellas-verona/2swm0r#4230531",
  "id": "4230531",
  "home": {
    "name": "Empoli",
    "shortName": "Empoli",
    "id": "8534"
  },
  "away": {
    "name": "Hellas Verona",
    "shortName": "Hellas Verona",
    "id": "9876"
  },
  "status": {
    "utcTime": "2023-08-19T16:30:00Z",
    "timezone": "",
    "finished": true,
    "started": true,
    "cancelled": false,
    "awarded": false,
    "scoreStr": "0 - 1",
    "reason": {
      "short": "FT",
      "shortKey": "fulltime_short",
      "long": "Full-Time",
      "longKey": "finished"
    }
  }
}


In [ ]:
matches_list = []
for m in all_matches:
    round_num = m.get("round", "")
    match_id = m.get("id", "")
    page_url = m.get("pageUrl", "")
    full_url = f"https://www.fotmob.com{page_url}" if page_url else ""
    
    home_name = m.get("home", {}).get("name", "")
    home_id = m.get("home", {}).get("id", "")
    away_name = m.get("away", {}).get("name", "")
    away_id = m.get("away", {}).get("id", "")
    
    status_obj = m.get("status", {})
    utc_time = status_obj.get("utcTime", "")
    score_str = status_obj.get("scoreStr", "")
    
    finished = status_obj.get("finished", False)
    cancelled = status_obj.get("cancelled", False)

    home_goals = ""
    away_goals = ""
    if score_str and " - " in score_str:
        try:
            parts = score_str.split(" - ")
            home_goals = int(parts[0])
            away_goals = int(parts[1])
        except:
            pass
            
    matches_list.append({
        "Giornata": int(round_num) if round_num.isdigit() else round_num,
        "ID Partita": match_id,
        "Data (UTC)": utc_time.replace("T", " ").replace("Z", ""),
        "Squadra Casa": home_name,
        "Squadra Fuori": away_name,
        "Risultato": score_str,
        "Gol Casa": home_goals,
        "Gol Fuori": away_goals,
        "Stato": "Finito" if finished else ("Cancellato" if cancelled else "Da giocare"),
        "URL FotMob": full_url
    })
print("\n")


for i in range(35, 40):
    print(matches_list[i], "\n")
print("\n")


df = pd.DataFrame(matches_list)
# Sort by round and then date/id
df = df.sort_values(by=["Giornata", "Data (UTC)"]).reset_index(drop=True)
print(df.head())
print("\n")


path = r".\Datasets\Matches\serieA_fixtures_2324.xlsx"
df.to_excel(path, index=False)



#creazione workbook con openpyxl
wb = openpyxl.Workbook()

ws_data = wb.active
ws_data.title = "Calendario e Risultati"
ws_data.views.sheetView[0].showGridLines = True



# Stile pagina
navy_fill = PatternFill(start_color="1B365D", end_color="1B365D", fill_type="solid")
light_blue_fill = PatternFill(start_color="F2F5F9", end_color="F2F5F9", fill_type="solid")
white_font = Font(name="Segoe UI", size=11, bold=True, color="FFFFFF")
regular_font = Font(name="Segoe UI", size=10)
bold_font = Font(name="Segoe UI", size=10, bold=True)
title_font = Font(name="Segoe UI", size=16, bold=True, color="1B365D")
section_font = Font(name="Segoe UI", size=12, bold=True, color="1B365D")

thin_gray = Side(style='thin', color='D3D3D3')
cell_border = Border(left=thin_gray, right=thin_gray, top=thin_gray, bottom=thin_gray)



# Populate Data Sheet
headers = list(df.columns)
ws_data.append(headers)
for row in ws_data[1]:
    row.fill = navy_fill
    row.font = white_font
    row.alignment = Alignment(horizontal="center", vertical="center")

for idx, r_data in df.iterrows():
    row_values = [r_data[h] for h in headers]
    ws_data.append(row_values)
    row_num = idx + 2
    
    # Apply link to URL FotMob column
    url_val = r_data["URL FotMob"]
    if url_val:
        cell = ws_data.cell(row=row_num, column=headers.index("URL FotMob") + 1)
        cell.hyperlink = url_val
        cell.font = Font(name="Segoe UI", size=10, color="0000FF", underline="single")
    
    # zebra striping
    if row_num % 2 == 1:
        for col_idx in range(1, len(headers) + 1):
            c = ws_data.cell(row=row_num, column=col_idx)
            if col_idx != headers.index("URL FotMob") + 1:
                c.fill = light_blue_fill

    # Borders and alignment
    for col_idx in range(1, len(headers) + 1):
        c = ws_data.cell(row=row_num, column=col_idx)
        c.border = cell_border
        if col_idx in [1, 2, 3, 6, 7, 8, 9]:
            c.alignment = Alignment(horizontal="center", vertical="center")
        else:
            c.alignment = Alignment(horizontal="left", vertical="center")

# Auto-adjust column widths
for col in ws_data.columns:
    max_len = max(len(str(cell.value or '')) for cell in col)
    col_letter = get_column_letter(col[0].column)
    ws_data.column_dimensions[col_letter].width = max(max_len + 3, 12)



ws_data.freeze_panes = "A2"
path_styled = r".\Datasets\Matches\Styled_SerieA_2324_Fixture_Stats.xlsx"
wb.save(path_styled)



{'Giornata': 4, 'ID Partita': '4230570', 'Data (UTC)': '2023-09-17 13:00:00', 'Squadra Casa': 'Monza', 'Squadra Fuori': 'Lecce', 'Risultato': '1 - 1', 'Gol Casa': 1, 'Gol Fuori': 1, 'Stato': 'Finito', 'URL FotMob': 'https://www.fotmob.com/matches/monza-vs-lecce/27zy6c#4230570'} 

{'Giornata': 4, 'ID Partita': '4230567', 'Data (UTC)': '2023-09-17 16:00:00', 'Squadra Casa': 'Fiorentina', 'Squadra Fuori': 'Atalanta', 'Risultato': '3 - 2', 'Gol Casa': 3, 'Gol Fuori': 2, 'Stato': 'Finito', 'URL FotMob': 'https://www.fotmob.com/matches/atalanta-vs-fiorentina/2en1cd#4230567'} 

{'Giornata': 4, 'ID Partita': '4230569', 'Data (UTC)': '2023-09-17 18:45:00', 'Squadra Casa': 'Roma', 'Squadra Fuori': 'Empoli', 'Risultato': '7 - 0', 'Gol Casa': 7, 'Gol Fuori': 0, 'Stato': 'Finito', 'URL FotMob': 'https://www.fotmob.com/matches/empoli-vs-roma/2ga6qg#4230569'} 

{'Giornata': 4, 'ID Partita': '4230566', 'Data (UTC)': '2023-09-18 16:30:00', 'Squadra Casa': 'Salernitana', 'Squadra Fuori': 'Torino', 'Ri

In [ ]:
# link API: https://www.fotmob.com/api/data/matchDetails?matchId={matchId}
print(headers)
print("\n")


matchId = df["ID Partita"]
print(matchId)
print("\n")


OUTPUT_DIR = r".\Datasets\MatchStatsJSON_2324"
os.makedirs(OUTPUT_DIR, exist_ok=True)



BASE_URL = "https://www.fotmob.com/api/data/matchDetails"
cookie_sporco = r"""g_state={"i_l":2,"i_ll":1779810410293,"i_b":"nRHE3uwL3TlzrPYvxMFcI0bZ+dl1r6S207EfAquto0Q","i_e":{"enable_itp_optimization":0},"i_et":1779710661468,"i_p":1779894495379}; _hjSessionUser_2585474=eyJpZCI6ImNlMmRkOTllLWE5NDMtNWFjYS05YzE3LTI2MTQxNTdhMTRlOSIsImNyZWF0ZWQiOjE3Nzk3MTA2NjE2MzQsImV4aXN0aW5nIjp0cnVlfQ==; FCCDCF=%5Bnull%2Cnull%2Cnull%2C%5B%22CQkw6oAQkw6oAEsACBITCgFoAP_gAEPgABpwLItB_C7dbWFDyLZ3absEeIhHx_hjasQwAAbAAmAFTBqQoJQGwmEYNAyCtCACGAAAKmTBIQIkCAAAAUAAIIAFIADMAAyQIBAIICBAiAEJAEBACEACCoAgUQCIgIJEFEQAm…rope%2FRome%22%7D; turnstile_verified=1.1779808090.d3c4b73a6c12d89300538a404f8022bc7f06d56f6306d6290bbe44a8a9fa453d; panoramaId_expiry=1779894492599; _hjSession_2585474=eyJpZCI6IjJkZjFlZjA1LTQ2NGItNDI4ZC1hYjAxLTdjOTMwODQzMDIxYyIsImMiOjE3Nzk4MTAzODQyODYsInMiOjAsInIiOjAsInNiIjowLCJzciI6MCwic2UiOjAsImZzIjowLCJzcCI6MX0=; FCNEC=%5B%5B%22AKsRol_4MENqJVHfam_1cJZuUm4Zg8gfe-eg-5W8vNrNNULYF96uETbLOxC-5Hg64DGPfoapHM-YN-5TSRXYQBMGcAJ_S-t8QowY8XoJ3jUYDjlXctL1CY5vfqJn3RcubAK4SnANnAe8ksOq_cBTCV9f51AQxRaYnA%3D%3D%22%5D%5D"""
cookie_pulito = clean_header_string(cookie_sporco)

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "it-IT,it;q=0.9,en-US;q=0.8,en;q=0.7",
    "Alt-Used": "www.fotmob.com",
    "Cookie": cookie_pulito
}



print("\n ============== INIZIO DOWNLOAD FILE JSON ==============")
for index, row in df.iterrows():
    match_id = str(row['ID Partita'])
    giornata = str(row['Giornata'])
    home_team = clean_filename(row['Squadra Casa'])
    away_team = clean_filename(row['Squadra Fuori'])
    
    # Pulizia della data
    full_date = clean_filename(str(row['Data (UTC)']))
    
    
    # Formato per salvare i file secondo l'ordine del file Excel
    file_name = f"{giornata}_{full_date}_{match_id}_{home_team}_{away_team}.json"
    file_path = os.path.join(OUTPUT_DIR, file_name)

    print(f"[{index + 1}/{len(df)}] Scaricamento: {home_team} vs {away_team} (ID: {match_id})")
    
    # Richiesta HTTP alla API
    params = {"matchId": match_id}
    try:
        response = requests.get(BASE_URL, params=params, headers=headers)
        
        if response.status_code == 200:
            match_json = response.json()
            
            # Salvataggio del file JSON
            with open(file_path, "w", encoding="utf-8") as out_file:
                import json
                json.dump(match_json, out_file, ensure_ascii=False, indent=4)
                
        elif response.status_code == 404:
            print(f"Dettagli non trovati (404) per il match {match_id}")
        else:
            print(f"Errore {response.status_code} per il match {match_id}")
            
    except Exception as e:
        print(f"Eccezione durante il download del match {match_id}: {e}")
        
    # Pausa di 1.5 secondi per evitare ban IP (per via delle tante richieste)
    time.sleep(1.5)

print("\n ============== FINE DOWNLOAD ==============")

['Giornata', 'ID Partita', 'Data (UTC)', 'Squadra Casa', 'Squadra Fuori', 'Risultato', 'Gol Casa', 'Gol Fuori', 'Stato', 'URL FotMob']


0      4230531
1      4230532
2      4230537
3      4230533
4      4230534
        ...   
375    4230913
376    4230906
377    4230907
378    4230908
379    4230910
Name: ID Partita, Length: 380, dtype: str



 ============== INIZIO DOWNLOAD FILE JSON ==============
[1/380] Scaricamento: Empoli vs Hellas_Verona (ID: 4230531)
[2/380] Scaricamento: Frosinone vs Napoli (ID: 4230532)
[3/380] Scaricamento: Genoa vs Fiorentina (ID: 4230537)
[4/380] Scaricamento: Inter vs Monza (ID: 4230533)
[5/380] Scaricamento: Roma vs Salernitana (ID: 4230534)
[6/380] Scaricamento: Sassuolo vs Atalanta (ID: 4230535)
[7/380] Scaricamento: Lecce vs Lazio (ID: 4230539)
[8/380] Scaricamento: Udinese vs Juventus (ID: 4230538)
[9/380] Scaricamento: Torino vs Cagliari (ID: 4230536)
[10/380] Scaricamento: Bologna vs Milan (ID: 4230530)
[11/380] Scaricamento: Frosinone vs Atalanta

In [ ]:
# lista vuota PRIMA del ciclo per accumulare i dati di tutte le partite
lista_partite_stats = []

print("============= INIZIO ELABORAZIONE TUTTI I FILE JSON ==============")

for index, row in df.iterrows():

    giornata = str(row["Giornata"])
    match_id = str(row["ID Partita"])
    full_date = str(row["Data (UTC)"])
    home_team = str(row["Squadra Casa"])
    away_team = str(row["Squadra Fuori"])
    result = str(row["Risultato"])
    home_goals = str(row['Gol Casa'])
    away_goals = str(row['Gol Fuori'])
    status = str(row["Stato"])
    link = str(row["URL FotMob"])

    # Pulizia per filename
    clean_home_team = clean_filename(home_team)
    clean_away_team = clean_filename(away_team)
    clean_full_date = clean_filename(full_date)

    # Formato del nome dei file
    file_name = f"{giornata}_{clean_full_date}_{match_id}_{clean_home_team}_{clean_away_team}.json"
    file_path = os.path.join(OUTPUT_DIR, file_name)

    print("Elaborazione file:", file_name)

    # check che il file esista
    if not os.path.exists(file_path):
        print(f"File non trovato per il match {match_id}. Salto.")
        continue

    # Apertura del file
    with open(file_path, "r", encoding="utf-8") as f:
        temp_data = json.load(f)

    # Inizializziamo il dizionario della partita corrente con i dati gia presenti
    dati_partita = {
        "Giornata": giornata,
        "ID Partita": match_id,
        "Data (UTC)": full_date,
        "Squadra Casa": home_team,
        "Squadra Fuori": away_team,
        "Risultato": result,
        "Gol Casa": home_goals,
        "Gol Fuori": away_goals,
        "Stato": status,
        "URL FotMob": link,
    }

    # Estrazione delle statistiche di squadra
    try:
        periods_stats = temp_data["content"]["stats"]["Periods"]["All"]["stats"]

        for category in periods_stats:
            sub_stats = category.get("stats", [])

            for stat in sub_stats:
                stat_title = stat.get("title")  # es. "Ball possession"
                values = stat.get("stats", [None, None])  # [Casa, Fuori]

                # Creiamo due colonne per differenziare i dati di Casa e Fuori
                dati_partita[f"{stat_title}_Casa"] = values[0]
                dati_partita[f"{stat_title}_Fuori"] = values[1]

    except KeyError:
        pass

    # Aggiungiamo le stats al totale
    lista_partite_stats.append(dati_partita)


print("============= FINE ELABORAZIONE TUTTI I FILE JSON ==============")




# creo dataframe e lo salvo in excel
df_statistiche_totale = pd.DataFrame(lista_partite_stats)

path_stats = r".\Datasets\MatchStatsTotal\SerieA_2324_Fixture_Stats.xlsx"
df_statistiche_totale.to_excel(path_stats, index=False)  




#creazione workbook con openpyxl
wb = openpyxl.Workbook()

ws_data = wb.active
ws_data.title = "Risultati e Statistiche"
ws_data.views.sheetView[0].showGridLines = True


# Stile pagina
navy_fill = PatternFill(start_color="1B365D", end_color="1B365D", fill_type="solid")
light_blue_fill = PatternFill(start_color="F2F5F9", end_color="F2F5F9", fill_type="solid")
white_font = Font(name="Segoe UI", size=11, bold=True, color="FFFFFF")
regular_font = Font(name="Segoe UI", size=10)
bold_font = Font(name="Segoe UI", size=10, bold=True)
title_font = Font(name="Segoe UI", size=16, bold=True, color="1B365D")
section_font = Font(name="Segoe UI", size=12, bold=True, color="1B365D")

thin_gray = Side(style='thin', color='D3D3D3')
cell_border = Border(left=thin_gray, right=thin_gray, top=thin_gray, bottom=thin_gray)


# Populate Data Sheet
headers = list(df_statistiche_totale.columns)
ws_data.append(headers)
for row in ws_data[1]:
    row.fill = navy_fill
    row.font = white_font
    row.alignment = Alignment(horizontal="center", vertical="center")

for idx, r_data in df_statistiche_totale.iterrows():
    row_values = [r_data[h] for h in headers]
    ws_data.append(row_values)
    row_num = idx + 2
    
    # Apply link to URL FotMob column
    url_val = r_data["URL FotMob"]
    if url_val:
        cell = ws_data.cell(row=row_num, column=headers.index("URL FotMob") + 1)
        cell.hyperlink = url_val
        cell.font = Font(name="Segoe UI", size=10, color="0000FF", underline="single")
    
    # Zebra striping
    if row_num % 2 == 1:
        for col_idx in range(1, len(headers) + 1):
            c = ws_data.cell(row=row_num, column=col_idx)
            if col_idx != headers.index("URL FotMob") + 1:
                c.fill = light_blue_fill

    # Borders and alignment
    for col_idx in range(1, len(headers) + 1):
        c = ws_data.cell(row=row_num, column=col_idx)
        c.border = cell_border
        if col_idx in [1, 2, 3, 6, 7, 8, 9]:
            c.alignment = Alignment(horizontal="center", vertical="center")
        else:
            c.alignment = Alignment(horizontal="left", vertical="center")

# Auto-adjust column widths
for col in ws_data.columns:
    max_len = max(len(str(cell.value or '')) for cell in col)
    col_letter = get_column_letter(col[0].column)
    ws_data.column_dimensions[col_letter].width = max(max_len + 3, 12)


ws_data.freeze_panes = "A2"
path_stats_styled = r".\Datasets\MatchStatsTotal\Styled_SerieA_2324_Fixture_Stats.xlsx"
wb.save(path_stats_styled)

============= INIZIO ELABORAZIONE TUTTI I FILE JSON ==============
Elaborazione file: 1_2023-08-19_163000_4230531_Empoli_Hellas_Verona.json
Elaborazione file: 1_2023-08-19_163000_4230532_Frosinone_Napoli.json
Elaborazione file: 1_2023-08-19_184500_4230537_Genoa_Fiorentina.json
Elaborazione file: 1_2023-08-19_184500_4230533_Inter_Monza.json
Elaborazione file: 1_2023-08-20_163000_4230534_Roma_Salernitana.json
Elaborazione file: 1_2023-08-20_163000_4230535_Sassuolo_Atalanta.json
Elaborazione file: 1_2023-08-20_184500_4230539_Lecce_Lazio.json
Elaborazione file: 1_2023-08-20_184500_4230538_Udinese_Juventus.json
Elaborazione file: 1_2023-08-21_163000_4230536_Torino_Cagliari.json
Elaborazione file: 1_2023-08-21_184500_4230530_Bologna_Milan.json
Elaborazione file: 2_2023-08-26_163000_4230547_Frosinone_Atalanta.json
Elaborazione file: 2_2023-08-26_163000_4230549_Monza_Empoli.json
Elaborazione file: 2_2023-08-26_184500_4230544_Milan_Torino.json
Elaborazione file: 2_2023-08-26_184500_4230543_Hell

# Now lets merge all three files in a single dataset
First of all we need to unify the features

In [ ]:
from itertools import zip_longest


def confronta_feature_colonne(df1, df2, df3):
    cols1 = df1.columns.tolist()
    cols2 = df2.columns.tolist()
    cols3 = df3.columns.tolist()

    # Stampare l'intestazione della tabella
    print(f"{'DF1':<25}{'DF2':<25}{'DF3':<25}")
    print("-" * 75)

    # zip_longest riempie i "vuoti" con una stringa vuota '' se un DF ha meno colonne degli altri
    for f1, f2, f3 in zip_longest(cols1, cols2, cols3, fillvalue=""):
        print(f"{str(f1):<25}{str(f2):<25}{str(f3):<25}")

In [ ]:
file_2324 = r".\Datasets\MatchStatsTotal\Styled_SerieA_2324_Fixture_Stats.xlsx"
file_2425 = r".\Datasets\MatchStatsTotal\Styled_SerieA_2425_Fixture_Stats.xlsx"
file_2526 = r".\Datasets\MatchStatsTotal\Styled_SerieA_2526_Fixture_Stats.xlsx"

print("Caricamento dei file in corso...")
df_2324 = pd.read_excel(file_2324)
df_2425 = pd.read_excel(file_2425)
df_2526 = pd.read_excel(file_2526)

# aggiungo una colonna per identificare la Stagione
df_2324["Stagione"] = "2023/2024"
df_2425["Stagione"] = "2024/2025"
df_2526["Stagione"] = "2025/2026"

confronta_feature_colonne(df_2324, df_2425, df_2526)


Caricamento dei file in corso...
DF1                      DF2                      DF3                      
---------------------------------------------------------------------------
Giornata                 Giornata                 Giornata                 
ID Partita               ID Partita               ID Partita               
Data (UTC)               Data (UTC)               Data (UTC)               
Squadra Casa             Squadra Casa             Squadra Casa             
Squadra Fuori            Squadra Fuori            Squadra Fuori            
Risultato                Risultato                Risultato                
Gol Casa                 Gol Casa                 Gol Casa                 
Gol Fuori                Gol Fuori                Gol Fuori                
Stato                    Stato                    Stato                    
URL FotMob               URL FotMob               URL FotMob               
Ball possession_Casa     Ball possession_Casa     Ball 

"In the first df, the Touches in opposition Box variable is located at the bottom of the dataset, while in the other two it is located about halfway through;<br> 
pandas should be able to handle it on its own, but just to be safe, I'll fix the rows manually.

In [ ]:
colonne_modello = df_2526.columns.tolist()
colonne_modello.remove("Stagione")
ordine_colonne_con_stagione_feat1 = ["Stagione"] + colonne_modello

# Forziamo il DF (23/24) e il DF2 (24/25) ad avere lo stesso identico ordine.
df_2324_ordinato = df_2324.reindex(columns=ordine_colonne_con_stagione_feat1)
df_2425_ordinato = df_2425.reindex(columns=ordine_colonne_con_stagione_feat1)
df_2526_ordinato = df_2526.reindex(columns=ordine_colonne_con_stagione_feat1)


print("--- VERIFICA ALLINEAMENTO COLONNE ---")
confronta_feature_colonne(
    df_2324_ordinato, df_2425_ordinato, df_2526_ordinato
)

--- VERIFICA ALLINEAMENTO COLONNE ---
DF1                      DF2                      DF3                      
---------------------------------------------------------------------------
Stagione                 Stagione                 Stagione                 
Giornata                 Giornata                 Giornata                 
ID Partita               ID Partita               ID Partita               
Data (UTC)               Data (UTC)               Data (UTC)               
Squadra Casa             Squadra Casa             Squadra Casa             
Squadra Fuori            Squadra Fuori            Squadra Fuori            
Risultato                Risultato                Risultato                
Gol Casa                 Gol Casa                 Gol Casa                 
Gol Fuori                Gol Fuori                Gol Fuori                
Stato                    Stato                    Stato                    
URL FotMob               URL FotMob               

In [ ]:
if list(df_2324_ordinato.columns) == list(df_2425_ordinato.columns) == list(df_2526_ordinato.columns):
    print("Confermato: Le colonne sono identiche e nello stesso ordine.")
else:
    print("Attenzione: C'è qualche discrepanza nei nomi o nell'ordine delle colonne.")
    print("Pandas prove a unirle per nome.")

# Unione dei file
print("\nUnione dei dataset...")
df_totale = pd.concat([df_2324_ordinato, df_2425_ordinato, df_2526_ordinato], ignore_index=True)

# Salvataggio
output_excel = r".\Datasets\SerieA_Fixture_Stats_Total.xlsx"
df_totale.to_excel(output_excel, index=False, sheet_name="Totale_23_26")

print("\n============== OPERAZIONE COMPLETATA ==============")
print(f"Righe totali generate: {len(df_totale)}")

Confermato: Le colonne sono identiche e nello stesso ordine.

Unione dei dataset...

============== OPERAZIONE COMPLETATA ==============
Righe totali generate: 1140


## Style for readability

In [ ]:
#creazione workbook con openpyxl
wb = openpyxl.Workbook()

ws_data = wb.active
ws_data.title = "Risultati e Statistiche"
ws_data.views.sheetView[0].showGridLines = True


# Stile pagina
navy_fill = PatternFill(start_color="1B365D", end_color="1B365D", fill_type="solid")
light_blue_fill = PatternFill(start_color="F2F5F9", end_color="F2F5F9", fill_type="solid")
white_font = Font(name="Segoe UI", size=11, bold=True, color="FFFFFF")
regular_font = Font(name="Segoe UI", size=10)
bold_font = Font(name="Segoe UI", size=10, bold=True)
title_font = Font(name="Segoe UI", size=16, bold=True, color="1B365D")
section_font = Font(name="Segoe UI", size=12, bold=True, color="1B365D")

thin_gray = Side(style='thin', color='D3D3D3')
cell_border = Border(left=thin_gray, right=thin_gray, top=thin_gray, bottom=thin_gray)


# Populate Data Sheet
headers = list(df_totale.columns)
ws_data.append(headers)
for row in ws_data[1]:
    row.fill = navy_fill
    row.font = white_font
    row.alignment = Alignment(horizontal="center", vertical="center")

for idx, r_data in df_totale.iterrows():
    row_values = [r_data[h] for h in headers]
    ws_data.append(row_values)
    row_num = idx + 2
    
    # Apply link to URL FotMob column
    url_val = r_data["URL FotMob"]
    if url_val:
        cell = ws_data.cell(row=row_num, column=headers.index("URL FotMob") + 1)
        cell.hyperlink = url_val
        cell.font = Font(name="Segoe UI", size=10, color="0000FF", underline="single")
    
    # Zebra striping
    if row_num % 2 == 1:
        for col_idx in range(1, len(headers) + 1):
            c = ws_data.cell(row=row_num, column=col_idx)
            if col_idx != headers.index("URL FotMob") + 1:
                c.fill = light_blue_fill

    # Borders and alignment
    for col_idx in range(1, len(headers) + 1):
        c = ws_data.cell(row=row_num, column=col_idx)
        c.border = cell_border
        if col_idx in [1, 2, 3, 6, 7, 8, 9]:
            c.alignment = Alignment(horizontal="center", vertical="center")
        else:
            c.alignment = Alignment(horizontal="left", vertical="center")

# Auto-adjust column widths
for col in ws_data.columns:
    max_len = max(len(str(cell.value or '')) for cell in col)
    col_letter = get_column_letter(col[0].column)
    ws_data.column_dimensions[col_letter].width = max(max_len + 3, 12)


ws_data.freeze_panes = "A2"
path_stats_styled = r".\Datasets\Styled_SerieA_Fixture_Stats_Total.xlsx"
wb.save(path_stats_styled)